# Lezione 4 — LLM generativi open source su GPU T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ccasadei-maggioli/corso-nlp-genai-2026/blob/main/lezione_4_llm_generativi/notebook_04_llm_generativi.ipynb)

Bentornato! 👋 Nelle lezioni precedenti abbiamo usato modelli "specializzati"
(sentiment, NER, classificazione): facevano *una cosa sola*. Oggi facciamo il salto
agli **LLM generativi**, modelli capaci di *scrivere* testo e seguire istruzioni in
linguaggio naturale. Li useremo sulle nostre **recensioni** per **riassumere**,
**estrarre pro e contro** e **rispondere a domande**.

In questa lezione:
1. capiamo *cos'è* un LLM generativo e come si controlla la generazione
   (**temperature**, **top_p**, **max_new_tokens**);
2. capiamo perché serve la **quantizzazione 4-bit** per far stare un modello 7B
   nella **GPU T4**;
3. carichiamo **`Qwen/Qwen2.5-7B-Instruct`** e impariamo il **chat template**
   (ruoli system/user/assistant);
4. mettiamo l'LLM al lavoro sulle recensioni con un po' di **prompting**.

> 🎯 **Filo conduttore:** sempre le stesse **recensioni clienti in italiano**.
> Oggi aggiungiamo la capacità di *generare* testo a partire da esse — il mattone
> su cui costruiremo, con LangChain (L5) e il RAG (L6), l'applicazione finale.

---
### ⚙️ Reminder: attiva la GPU T4
Menu **`Runtime` → `Change runtime type` → Hardware accelerator: `T4 GPU` → `Save`**.
Oggi la GPU è **indispensabile**: senza, il modello non si carica.

> ⏳ **AVVISO sui tempi:** un modello **7B in 4-bit**
> sulla T4 è sorprendentemente capace, ma **non** è istantaneo: ogni risposta può
> richiedere **qualche secondo** (a volte 10-30s per i riassunti più lunghi). È
> normale. Più avanti trovi consigli su come gestire queste attese e un'alternativa
> più leggera (Qwen2.5-**3B**) se la T4 risulta lenta.

## 1. Installiamo le librerie

Oltre a `transformers`, oggi servono due librerie chiave:
- **`accelerate`** — gestisce il posizionamento del modello sulla GPU (`device_map`);
- **`bitsandbytes`** — implementa la **quantizzazione 4-bit**, cioè il "trucco" che
  permette a un modello da 7 miliardi di parametri di entrare nei ~15 GB della T4.

> 💡 Come sempre, ogni notebook installa da sé ciò che gli serve, così puoi aprire
> questa lezione in modo indipendente.

In [ ]:
# -q = silenzioso. La prima installazione richiede un minuto circa.
!pip install -q "transformers>=4.45" "accelerate>=0.34" "bitsandbytes>=0.44"
print("Librerie installate ✅")

In [ ]:
import torch

print("Versione PyTorch:", torch.__version__)
print("GPU disponibile:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Scheda:", torch.cuda.get_device_name(0))
else:
    print("⚠️  GPU non attiva. Vai su Runtime > Change runtime type > T4 GPU,")
    print("    altrimenti il modello 7B NON si caricherà.")

## 2. Che cos'è un LLM generativo? 🧠

Un **LLM generativo** (Large Language Model) è, in fondo, una macchina che fa una cosa
sola, ma molto bene: **predice il token successivo**. Gli diamo un testo (il *prompt*)
e lui calcola, per ogni possibile token del suo vocabolario, una probabilità; ne sceglie
uno, lo aggiunge al testo e ripete. Frase dopo frase, nasce così una risposta.

Da questo "predire il prossimo token" emergono capacità sorprendenti: riassumere,
rispondere a domande, riscrivere, tradurre… **senza essere addestrato apposta** per
ciascun compito. Glielo chiediamo a parole, ed è questo il cuore della *Generative AI*.

### I parametri di sampling (come controlliamo la generazione)
Quando il modello sceglie il token successivo, *come* sceglie tra i candidati è regolabile:

- **`temperature`** — quanto "rischia". A **0.0** sceglie sempre il token più probabile
  → output **deterministico** e prevedibile (ideale per estrazione/classificazione).
  Salendo (es. **0.7–0.9**) le scelte diventano più varie e **creative**.
- **`top_p`** (nucleus sampling) — limita la scelta ai token più probabili la cui
  probabilità cumulata raggiunge `p` (es. **0.9**). Taglia la "coda" di token improbabili
  e bizzarri.
- **`max_new_tokens`** — **quanti** token al massimo generare. È il principale freno alla
  *lunghezza* (e quindi al *tempo*) della risposta. Più alto = risposte più lunghe e lente.

> 🔑 Regola pratica: **estrarre/classificare → `temperature=0`**; **scrivere testo per
> persone → `temperature≈0.7`**.

## 3. La quantizzazione 4-bit: far stare un 7B nella T4 📦

Un modello con **7 miliardi** di parametri, in precisione `float16` (2 byte per
parametro), occupa **~14 GB** solo di pesi. La GPU **T4** ha **~15 GB** di memoria: ci
starebbe a malapena, senza spazio per l'elaborazione → andrebbe in *out of memory*.

La **quantizzazione 4-bit** comprime ogni peso da 16 a **4 bit**. Risultato:

| Formato | Byte per parametro | Peso di un 7B | Sta nella T4? |
|---|---|---|---|
| `float16` (fp16) | 2 | ~14 GB | ❌ a malapena, poi OOM |
| **4-bit (nf4)** | 0.5 | **~5–6 GB** | ✅ con margine |

Si perde un filo di precisione, ma la qualità resta **ottima** per i nostri scopi, e in
cambio il modello **entra** nella GPU con spazio per lavorare. È il compromesso che rende
gli LLM open accessibili senza hardware costoso.

### Il chat template (ruoli system / user / assistant) 💬
I modelli *Instruct* come Qwen sono addestrati a dialogare. Non gli passiamo testo grezzo,
ma una lista di **messaggi** con un **ruolo**:

- **`system`** — le istruzioni di fondo: chi è l'assistente, in che lingua e stile risponde
  («Sei un assistente che analizza recensioni e risponde in italiano…»);
- **`user`** — la richiesta dell'utente (il nostro compito + i dati);
- **`assistant`** — la risposta del modello.

Il **chat template** del tokenizer trasforma questa lista nel formato esatto (con i token
speciali) che il modello si aspetta. Lo useremo tra poco con `apply_chat_template`.

## 4. Il nostro dataset: recensioni clienti 🛒

Ricarichiamo le solite recensioni sintetiche in italiano (schema:
`id, data, prodotto, categoria, rating, titolo, testo`). Sono *riproducibili* (seed fisso)
e non richiedono download esterni.

In [ ]:
import os, torch
import pandas as pd

# Scarica lo script generatore se non è già nella sessione Colab.
if not os.path.exists("genera_recensioni.py"):
    !wget -q https://raw.githubusercontent.com/ccasadei-maggioli/corso-nlp-genai-2026/main/dati/genera_recensioni.py

import genera_recensioni

df = pd.DataFrame(genera_recensioni.genera_recensioni(n=200, seed=42))
print("Numero di recensioni:", len(df))
df.head(3)

## 5. Carichiamo l'LLM in 4-bit 🚀

Carichiamo **`Qwen/Qwen2.5-7B-Instruct`**: un modello open (licenza Apache-2.0), molto
forte in italiano, nella sua versione *Instruct* (addestrata a seguire istruzioni).

> ⏳ La **prima** esecuzione **scarica** il modello (qualche GB): può richiedere **1–3
> minuti**. Le esecuzioni successive nella stessa sessione sono immediate. È il momento
> giusto per una pausa...

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = "Qwen/Qwen2.5-7B-Instruct"

# Configurazione della quantizzazione 4-bit (gestita da bitsandbytes).
bnb = BitsAndBytesConfig(
    load_in_4bit=True,              # carica i pesi a 4 bit invece di 16 (~14GB -> ~5-6GB)
    bnb_4bit_quant_type="nf4",      # "NormalFloat4": schema 4-bit ottimizzato per i pesi
    bnb_4bit_compute_dtype=torch.float16,  # i CALCOLI avvengono in fp16 (qualità migliore)
    bnb_4bit_use_double_quant=True, # quantizza anche le costanti di quant. (risparmia altra RAM)
)

# Il tokenizer trasforma testo <-> token e contiene il chat template del modello.
tokenizer = AutoTokenizer.from_pretrained(model_id)

# device_map="auto": accelerate posiziona automaticamente il modello sulla GPU.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb,
    device_map="auto",
)

print("Modello caricato ✅  Memoria GPU usata: "
      f"{torch.cuda.memory_allocated() / 1e9:.1f} GB")

> 💡 **Alternativa più leggera/veloce.** Se la T4 ti sembra lenta (o vai in *out of
> memory*), passa a **`Qwen/Qwen2.5-3B-Instruct`**: basta cambiare la riga
> `model_id = "Qwen/Qwen2.5-3B-Instruct"` qui sopra e rieseguire la cella. È un modello
> più piccolo: risponde **più in fretta**, con qualità un po' inferiore ma del tutto
> adeguata per gli esempi di questa lezione.

## 6. Una funzione di generazione comoda 🛠️

Per non ripetere lo stesso codice, definiamo una funzione `genera(messaggi)` che:
1. applica il **chat template** alla lista di messaggi (ruoli system/user/assistant);
2. tokenizza e sposta i dati sulla GPU;
3. chiama `model.generate(...)` con i parametri di sampling;
4. **decodifica solo i token nuovi** (la risposta), scartando il prompt iniziale.

In [ ]:
def genera(messaggi, max_new_tokens=256, temperature=0.7):
    # 1) Trasforma la lista di messaggi nel formato atteso dal modello.
    #    add_generation_prompt=True aggiunge l'innesco per la risposta dell'assistant.
    testo = tokenizer.apply_chat_template(messaggi, tokenize=False, add_generation_prompt=True)

    # 2) Tokenizza e porta i tensori sullo stesso device del modello (la GPU).
    inputs = tokenizer(testo, return_tensors="pt").to(model.device)

    # 3) Genera. do_sample=False (cioè temperature=0) -> output deterministico.
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,        # se temperature=0 -> greedy (deterministico)
        temperature=temperature,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

    # 4) Tieni solo i token GENERATI (dopo il prompt) e decodificali in testo.
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


# Prova al volo.
messaggi = [
    {"role": "system", "content": "Sei un assistente conciso che risponde in italiano."},
    {"role": "user", "content": "In una frase, a cosa serve l'analisi delle recensioni clienti?"},
]
print(genera(messaggi, max_new_tokens=80))

## 7. Esempio 1 — Riassunto di più recensioni 📝

Scenario tipico: un prodotto ha **decine** di recensioni e vogliamo il succo in **3 punti**.
Prendiamo tutte le recensioni di un prodotto, le impacchettiamo nel messaggio `user` e
chiediamo all'LLM un riassunto. Per un riassunto "creativo ma fedele" va bene
`temperature≈0.5`.

In [ ]:
# Scegliamo un prodotto e raccogliamo le sue recensioni.
prodotto = "Cuffie Bluetooth XSound Pro"
recensioni_prod = df[df["prodotto"] == prodotto]
print(f"{prodotto}: {len(recensioni_prod)} recensioni\n")

# Le concateniamo in un blocco di testo (limitiamo a 12 per non allungare troppo il prompt).
blocco = "\n".join(
    f"- [{r.rating}★] {r.testo}" for r in recensioni_prod.head(12).itertuples()
)

messaggi = [
    {"role": "system", "content":
        "Sei un analista che sintetizza recensioni clienti. Rispondi sempre in italiano, "
        "in modo conciso e fedele ai contenuti, senza inventare informazioni."},
    {"role": "user", "content":
        f"Ecco le recensioni del prodotto '{prodotto}':\n\n{blocco}\n\n"
        "Riassumi l'opinione complessiva dei clienti in 3 punti elenco."},
]

print(genera(messaggi, max_new_tokens=220, temperature=0.5))

## 8. Esempio 2 — Estrazione di pro e contro ✅❌

Qui non vogliamo prosa, ma un **output strutturato**: due elenchi, *Pro* e *Contro*.
È un compito di **estrazione**: vogliamo che il modello sia fedele e ripetibile, quindi
abbassiamo la `temperature` a **0.2**.

In [ ]:
messaggi = [
    {"role": "system", "content":
        "Sei un assistente che estrae informazioni dalle recensioni. Rispondi in italiano "
        "e attieniti SOLO a ciò che è scritto nelle recensioni."},
    {"role": "user", "content":
        f"Dalle seguenti recensioni del prodotto '{prodotto}':\n\n{blocco}\n\n"
        "Elenca i PRO e i CONTRO principali. Usa esattamente questo formato:\n"
        "PRO:\n- ...\nCONTRO:\n- ..."},
]

print(genera(messaggi, max_new_tokens=220, temperature=0.2))

## 9. Esempio 3 — Rispondere a una domanda su una recensione ❓

Diamo al modello **una** recensione e gli poniamo una **domanda specifica** su di essa.
È l'idea alla base del Q&A sui documenti che, su larga scala, diventerà il **RAG** della
Lezione 6. Qui la risposta deve basarsi solo sul testo fornito: `temperature=0`.

In [ ]:
# Prendiamo una recensione lunga (con più aspetti) come esempio.
recensione = df.iloc[1]
print(f"Recensione [{recensione['rating']}★] su '{recensione['prodotto']}':")
print(recensione["testo"], "\n")

messaggi = [
    {"role": "system", "content":
        "Sei un assistente che risponde a domande basandoti SOLO sul testo fornito. "
        "Rispondi in italiano. Se l'informazione non è presente, dillo chiaramente."},
    {"role": "user", "content":
        f"Recensione:\n\"{recensione['testo']}\"\n\n"
        "Domanda: il cliente è soddisfatto della spedizione? Rispondi in una frase, "
        "citando la parte di testo che lo conferma."},
]

print("Risposta:", genera(messaggi, max_new_tokens=120, temperature=0.0))

## 10. Prompting: prompt vago vs prompt ben fatto ✍️

La **qualità della risposta dipende enormemente dalla qualità del prompt.** Confrontiamo
due modi di chiedere la stessa cosa:

- **Vago:** una richiesta generica, senza ruolo né istruzioni precise.
- **Ben fatto:** un `system` chiaro + istruzioni esplicite su *cosa*, *come* e *in che
  formato* rispondere.

Stesso modello, stessa recensione: cambia solo *come* chiediamo.

In [ ]:
testo_rec = df.iloc[1]["testo"]

# --- Prompt VAGO ---
vago = [
    {"role": "user", "content": f"{testo_rec}\nCosa ne pensi?"},
]
print("=== PROMPT VAGO ===")
print(genera(vago, max_new_tokens=160, temperature=0.7))

In [ ]:
# --- Prompt BEN FATTO ---
ben_fatto = [
    {"role": "system", "content":
        "Sei un analista di customer experience. Rispondi in italiano, in modo strutturato "
        "e basandoti solo sul testo della recensione."},
    {"role": "user", "content":
        f"Recensione del cliente:\n\"{testo_rec}\"\n\n"
        "Produci un'analisi in 3 righe:\n"
        "1) Sentiment complessivo (positivo/neutro/negativo).\n"
        "2) Aspetti citati (es. spedizione, prezzo, qualità...).\n"
        "3) Una possibile azione per l'azienda."},
]
print("=== PROMPT BEN FATTO ===")
print(genera(ben_fatto, max_new_tokens=200, temperature=0.3))

## 11. L'effetto della `temperature` 🌡️

Stesso prompt, due `temperature` diverse. Chiediamo una frase di ringraziamento a un
cliente. Esegui la cella **più volte**:

- con **`temperature=0.0`** la risposta è **sempre identica** (deterministica);
- con **`temperature=0.9`** cambia ogni volta, più **varia e creativa**.

In [ ]:
prompt_creativo = [
    {"role": "system", "content": "Sei l'assistenza clienti di un e-commerce. Scrivi in italiano."},
    {"role": "user", "content":
        "Scrivi UNA frase per ringraziare un cliente che ha lasciato una recensione a 5 stelle."},
]

print("--- temperature = 0.0 (deterministico) ---")
print(genera(prompt_creativo, max_new_tokens=60, temperature=0.0))

print("\n--- temperature = 0.9 (creativo) ---")
print(genera(prompt_creativo, max_new_tokens=60, temperature=0.9))

## 12. Esercizio 🏋️

Tocca a te. Scegli un prodotto (puoi cambiarlo) e scrivi un **prompt** che, date le sue
recensioni, faccia produrre all'LLM una **risposta dell'assistenza clienti**: un breve
messaggio (3-4 frasi) che ringrazia per i feedback positivi e rassicura sui punti critici
emersi, in tono cortese e professionale.

Completa il `system` e il `user` dove indicato dai **TODO**, poi esegui.

> 💡 *Variante alternativa:* invece della risposta dell'assistenza, chiedi di estrarre
> i **pro/contro** in formato elenco (come nell'Esempio 2) ma per un prodotto diverso.

In [ ]:
prodotto_es = "Smartwatch FitPro 2"
recensioni_es = df[df["prodotto"] == prodotto_es]
blocco_es = "\n".join(
    f"- [{r.rating}★] {r.testo}" for r in recensioni_es.head(12).itertuples()
)

messaggi_es = [
    # TODO: scrivi un buon system prompt. Suggerimento: definisci il ruolo
    #       ("addetto all'assistenza clienti"), la lingua (italiano) e il tono
    #       (cortese, professionale).
    {"role": "system", "content": "..."},
    # TODO: nel messaggio user, inserisci il blocco di recensioni (blocco_es) e
    #       chiedi una risposta dell'assistenza di 3-4 frasi che ringrazi per i
    #       feedback positivi e rassicuri sui punti critici.
    {"role": "user", "content": "..."},
]

# print(genera(messaggi_es, max_new_tokens=220, temperature=0.6))

### ✅ Soluzione

Una possibile soluzione (la tua può essere diversa: ciò che conta è un `system` chiaro e
istruzioni esplicite nel `user`).

In [ ]:
messaggi_sol = [
    {"role": "system", "content":
        "Sei un addetto all'assistenza clienti di un e-commerce. Rispondi sempre in "
        "italiano, con tono cortese e professionale. Ti basi solo sui contenuti delle "
        "recensioni e non inventi dettagli."},
    {"role": "user", "content":
        f"Ecco le recensioni del prodotto '{prodotto_es}':\n\n{blocco_es}\n\n"
        "Scrivi una risposta pubblica dell'assistenza clienti di 3-4 frasi: ringrazia per "
        "i feedback positivi e rassicura i clienti sui principali punti critici emersi, "
        "indicando che li terremo in considerazione."},
]

print(genera(messaggi_sol, max_new_tokens=220, temperature=0.6))

## 13. Riepilogo e prossimi passi ✅

Oggi abbiamo fatto il salto agli **LLM generativi**:
- capito che un LLM **predice il token successivo** e si controlla con
  **`temperature`**, **`top_p`** e **`max_new_tokens`**;
- usato la **quantizzazione 4-bit** per far stare **Qwen2.5-7B** nella **GPU T4**
  (~5-6 GB invece di ~14);
- imparato il **chat template** (ruoli `system`/`user`/`assistant`) e scritto una
  funzione `genera(...)` riutilizzabile;
- messo l'LLM al lavoro sulle recensioni: **riassunto**, **pro/contro**, **Q&A**, e
  visto la differenza tra prompt **vago** e prompt **ben fatto**, oltre all'effetto della
  `temperature`.

Abbiamo anche visto l'**alternativa leggera** `Qwen/Qwen2.5-3B-Instruct` per quando la
T4 è lenta.

➡️ **Prossima lezione (LangChain):** finora abbiamo scritto i prompt "a mano" e incollato
i dati nelle stringhe. Nella **Lezione 5** impareremo a **strutturare** tutto questo con
**LangChain**: *prompt template* riutilizzabili, **catene** (LCEL) che concatenano i passi,
**parser** per ottenere output strutturato e **memoria** per il dialogo — tutto attorno a
questo stesso LLM. È il ponte verso l'applicazione finale con **RAG** (L6).

📦 Tutto il materiale del corso: https://github.com/ccasadei-maggioli/corso-nlp-genai-2026